# Notebook 02: Silver Transformation (Entity Resolution & Sentiment Analysis)

Transforms raw Reddit comments and player rosters into a structured, enriched Silver Delta table:
1. **Entity Resolution**: Matches player mentions in comment bodies using exact word boundary matching and fuzzy fallback (RapidFuzz).
2. **Sentiment Analysis**: Evaluates toxicity and sentiment per comment using VADER sentiment analysis via distributed PySpark `pandas_udf`.
3. **Output**: Writes `performance_vs_toxicity.silver.tagged_comments` in Delta Lake format.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is available on sys.path dynamically
repo_root = str(Path(os.getcwd()).resolve())
if repo_root not in sys.path:
    sys.path.append(repo_root)

import pandas as pd
from pyspark.sql.functions import explode, lit, pandas_udf
from pyspark.sql.types import ArrayType, IntegerType

from src.common.config import load_config
from src.entity_resolution.player_matcher import build_alias_index, match_players
from src.classification.sentiment_baseline import score_comment

cfg = load_config()
RAW_DIR = "/Volumes/performance_vs_toxicity/bronze/raw_files"

all_seasons_df = None

for season_id in cfg["seasons"]:
    # Respectful handling / specific exclusion if needed
    EXCLUDED_PLAYERS = {"Diogo J."}

    # Custom aliases to capture fan terminology not covered by default FPL web_names
    CUSTOM_ALIASES = {
        "A.Becker": ["alisson"],
        "Luis Díaz": ["diaz", "díaz"],
    }

    players_df = pd.read_csv(f"{RAW_DIR}/fpl/{season_id}/players.csv")
    players_df = players_df[~players_df["web_name"].isin(EXCLUDED_PLAYERS)]
    alias_index = build_alias_index(players_df, custom_aliases=CUSTOM_ALIASES)

    names_pd = pd.DataFrame({
        "player_id": list(alias_index.display_name.keys()),
        "player_name": list(alias_index.display_name.values()),
    })
    names_df = spark.createDataFrame(names_pd)

    def make_match_udf(idx):
        @pandas_udf(ArrayType(IntegerType()))
        def _match(text: pd.Series) -> pd.Series:
            return text.fillna("").apply(lambda t: match_players(t, idx))
        return _match

    @pandas_udf("compound double, label string")
    def sentiment_udf(text: pd.Series) -> pd.DataFrame:
        scores = text.fillna("").apply(score_comment)
        return pd.DataFrame({
            "compound": scores.apply(lambda s: s["compound"]),
            "label": scores.apply(lambda s: s["label"]),
        })

    match_udf = make_match_udf(alias_index)
    raw = spark.read.json(f"{RAW_DIR}/reddit/{season_id}/comments.jsonl")

    tagged = (
        raw
        .withColumn("player_ids", match_udf("body"))
        .withColumn("sentiment", sentiment_udf("body"))
        .filter("size(player_ids) > 0")
        .select(
            raw["id"].alias("comment_id"),
            "created_utc",
            "score",
            explode("player_ids").alias("player_id"),
            "sentiment.compound",
            "sentiment.label"
        )
        .withColumnRenamed("compound", "sentiment_compound")
        .withColumnRenamed("label", "sentiment_label")
        .join(names_df, on="player_id", how="left")
        .withColumn("season", lit(season_id))
    )

    all_seasons_df = tagged if all_seasons_df is None else all_seasons_df.unionByName(tagged)

# Write transformed data to Silver Delta table
all_seasons_df.write.format("delta").mode("overwrite") \
    .saveAsTable("performance_vs_toxicity.silver.tagged_comments")
print("Successfully written Silver Delta table: performance_vs_toxicity.silver.tagged_comments")